# 🧪 Week 3 — Day 1: Feature Engineering
### Applied AI & LLM Engineer | Edversity

---

**What is this notebook?**  
This notebook teaches you how to prepare your raw data so that machine-learning models can actually learn from it. That process is called **Feature Engineering**.

**What will we do today?**
1. Encode categorical (text) columns into numbers
2. Scale numerical columns so they are on the same range
3. Select only the most useful features
4. Handle imbalanced datasets with SMOTE

> 💡 **Tip:** Run each cell one at a time with **Shift + Enter**. Read the explanation above the cell *before* running it.


---
## 🔧 Step 0 — Install & Import Libraries

Before we can use any tools, we need to:
1. **Install** any missing packages (done once with `pip install`)
2. **Import** the libraries into our Python session

Think of libraries as **toolboxes** — `pandas` is your data table tool, `sklearn` is your ML tool, and `imbalanced-learn` gives us SMOTE.


In [6]:
# Install imbalanced-learn (only needed once — safe to run again)
# The ! at the start tells the notebook to run a terminal command
!pip install imbalanced-learn --quiet


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
# ── Standard libraries ──────────────────────────────────────────────────────
import pandas as pd          # for creating and manipulating data tables (DataFrames)
import numpy as np           # for numerical operations

# ── Encoding ────────────────────────────────────────────────────────────────
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, StandardScaler, MinMaxScaler, RobustScaler

# ── Feature Selection ───────────────────────────────────────────────────────
from sklearn.feature_selection import SelectKBest, f_classif, RFE, SelectFromModel
from sklearn.ensemble import RandomForestClassifier

# ── Datasets & Evaluation ───────────────────────────────────────────────────
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# ── Imbalanced data ─────────────────────────────────────────────────────────
from imblearn.over_sampling import SMOTE

print("✅ All libraries imported successfully!")


✅ All libraries imported successfully!


---
## 📦 Section 1 — Encoding Categorical Variables

### Why do we need encoding?

Machine learning models are **pure mathematics** — they can only work with **numbers**.  
But real-world data often contains text like `"Male"`, `"Karachi"`, or `"PhD"`.

We must convert text → numbers. This is called **encoding**.

There are three main methods:

| Method | When to use it | Example |
|--------|---------------|---------|
| **Label Encoding** | When the column has a natural order | `Bachelors=0, Masters=1, PhD=2` |
| **One-Hot Encoding** | When there is NO natural order | `city_Lahore=1, city_Islamabad=0` |
| **Ordinal Encoding** | When you want to define the order yourself | Same as Label, but you control it |

Let's build a small example dataset first.


In [8]:

df = pd.DataFrame({
    'city':      ['Karachi', 'Lahore', 'Islamabad', 'Lahore', 'Karachi'],
    'education': ['PhD',     'Masters', 'Bachelors', 'Masters', 'PhD'],
    'gender':    ['Male',    'Female',  'Male',      'Female',  'Male']
})
df

,city,education,gender
0,Karachi,PhD,Male
1,Lahore,Masters,Female
2,Islamabad,Bachelors,Male
3,Lahore,Masters,Female
4,Karachi,PhD,Male


In [20]:
# ── Create a tiny example dataset ──────────────────────────────────────────
# This is NOT real survey data — we made it up to practice encoding.

df = pd.DataFrame({
    'city':      ['Karachi', 'Lahore', 'Islamabad', 'Lahore', 'Karachi'],
    'education': ['PhD',     'Masters', 'Bachelors', 'Masters', 'PhD'],
    'gender':    ['Male',    'Female',  'Male',      'Female',  'Male']
})

print("Our raw dataset (text columns):")
print(df)
print()
print("Column data types:")
print(df.dtypes)


Our raw dataset (text columns):
        city  education  gender
0    Karachi        PhD    Male
1     Lahore    Masters  Female
2  Islamabad  Bachelors    Male
3     Lahore    Masters  Female
4    Karachi        PhD    Male

Column data types:
city         str
education    str
gender       str
dtype: object


In [21]:
'Bachelor'=='Bachelor'

True

In [22]:
df['education'].unique()

<StringArray>
['PhD', 'Masters', 'Bachelors']
Length: 3, dtype: str

### 1.1 — Label Encoding (for columns with a natural order)

`LabelEncoder` converts each unique text value to an integer.  
**Order is decided alphabetically by default**, which is why we use it on `education` where there is a natural ordering (but we'll fix the order in 1.3).

> ⚠️ Do NOT use Label Encoding on city names — the model might wrongly think Lahore (1) is "greater than" Islamabad (0).


In [23]:
# ── Method 1: Label Encoding ────────────────────────────────────────────────
# sklearn's LabelEncoder fits (learns the unique values) then transforms (converts)

le = LabelEncoder()

# fit_transform() does two things at once:
#   fit()       → learn: "what unique values exist?"
#   transform() → apply: "replace each value with its number"
df['education_label_encoded'] = le.fit_transform(df['education'])

print("After Label Encoding on 'education':")
print(df[['education', 'education_label_encoded']])
print()

# See the mapping: which number stands for which label?
print("Mapping (index = number):", list(le.classes_))
print("  → Bachelors=0, Masters=1, PhD=2  (alphabetical order)")


After Label Encoding on 'education':
   education  education_label_encoded
0        PhD                        2
1    Masters                        1
2  Bachelors                        0
3    Masters                        1
4        PhD                        2

Mapping (index = number): ['Bachelors', 'Masters', 'PhD']
  → Bachelors=0, Masters=1, PhD=2  (alphabetical order)


### 1.2 — One-Hot Encoding (for columns WITHOUT a natural order)

`pd.get_dummies()` creates **one new column per unique city**.  
Each row gets a `1` in its city's column and `0` everywhere else.

We use `drop_first=True` to avoid the **dummy variable trap** (multicollinearity).  
If we know `city_Islamabad=0` and `city_Lahore=0`, we already know the city is Karachi — so we don't need a `city_Karachi` column.


In [24]:
# ── Method 2: One-Hot Encoding ──────────────────────────────────────────────
# pd.get_dummies creates one binary (0/1) column per unique category

city_dummies = pd.get_dummies(df['city'], prefix='city', drop_first=True)

print("New 'one-hot' columns for city:")
print(city_dummies)
print()

# Join (concatenate) these new columns onto our main dataframe
df = pd.concat([df, city_dummies], axis=1)

print("DataFrame now includes one-hot city columns:")
print(df[['city', 'city_Karachi', 'city_Lahore']].head())


New 'one-hot' columns for city:
   city_Karachi  city_Lahore
0          True        False
1         False         True
2         False        False
3         False         True
4          True        False

DataFrame now includes one-hot city columns:
        city  city_Karachi  city_Lahore
0    Karachi          True        False
1     Lahore         False         True
2  Islamabad         False        False
3     Lahore         False         True
4    Karachi          True        False


In [33]:
df['education'].unique()

<StringArray>
['PhD', 'Masters', 'Bachelors']
Length: 3, dtype: str

### 1.3 — Ordinal Encoding (you define the order)

`OrdinalEncoder` is like Label Encoding, but **you tell it the correct order**.  
This is safer than `LabelEncoder` when the alphabetical order would be wrong.


In [29]:
# ── Method 3: Ordinal Encoding ──────────────────────────────────────────────
# We explicitly say: Bachelors < Masters < PhD

oe = OrdinalEncoder(categories=[['Bachelors', 'Masters', 'PhD']])

# Note: OrdinalEncoder expects a 2D input, so we use double brackets df[['education']]
df['education_ordinal'] = oe.fit_transform(df[['education']])

print("After Ordinal Encoding on 'education':")
print(df[['education', 'education_ordinal']])
print()
print("Bachelors → 0.0,  Masters → 1.0,  PhD → 2.0  ✅")


After Ordinal Encoding on 'education':
   education  education_ordinal
0        PhD                2.0
1    Masters                1.0
2  Bachelors                0.0
3    Masters                1.0
4        PhD                2.0

Bachelors → 0.0,  Masters → 1.0,  PhD → 2.0  ✅


---
## 📏 Section 2 — Feature Scaling

### Why do we need scaling?

Imagine two columns: `age` (range 20–45) and `salary` (range 30,000–150,000).

A model that uses distance (like K-Nearest Neighbours) will almost completely ignore `age`  
because a difference of `1 year` is tiny compared to a salary gap of `10,000`.

**Scaling puts all features on a comparable range.**

| Scaler | What it does | Best for |
|--------|-------------|---------|
| **StandardScaler** | Makes mean=0, std=1 | Algorithms that assume normal distribution (Logistic Regression, SVM) |
| **MinMaxScaler** | Squashes to [0, 1] | Neural Networks |
| **RobustScaler** | Uses median (not mean) | Data with outliers |


In [34]:
# ── Create a tiny example dataset ──────────────────────────────────────────
data = np.array([
    [25,  40000],
    [30,  60000],
    [35,  80000],
    [22,  30000],
    [45, 150000]   # ← salary outlier
])
df_scale = pd.DataFrame(data, columns=['age', 'salary'])

print("Original data:")
print(df_scale)


Original data:
   age  salary
0   25   40000
1   30   60000
2   35   80000
3   22   30000
4   45  150000


In [35]:
# ── StandardScaler ──────────────────────────────────────────────────────────
# Subtracts the mean and divides by the standard deviation.
# Formula: z = (x - mean) / std

ss = StandardScaler()
df_scale['age_standard'] = ss.fit_transform(df_scale[['age']])

print("Age after StandardScaler (mean≈0, spread≈1):")
print(df_scale[['age', 'age_standard']].round(3))
print(f"  New mean: {df_scale['age_standard'].mean():.4f}  (should be ~0)")
print(f"  New std:  {df_scale['age_standard'].std():.4f}   (should be ~1)")


Age after StandardScaler (mean≈0, spread≈1):
   age  age_standard
0   25        -0.789
1   30        -0.173
2   35         0.444
3   22        -1.158
4   45         1.676
  New mean: 0.0000  (should be ~0)
  New std:  1.1180   (should be ~1)


In [36]:
# ── MinMaxScaler ────────────────────────────────────────────────────────────
# Scales all values to the [0, 1] range.
# Formula: x_scaled = (x - min) / (max - min)

mm = MinMaxScaler()
df_scale['salary_minmax'] = mm.fit_transform(df_scale[['salary']])

print("Salary after MinMaxScaler (range 0 to 1):")
print(df_scale[['salary', 'salary_minmax']].round(3))


Salary after MinMaxScaler (range 0 to 1):
   salary  salary_minmax
0   40000          0.083
1   60000          0.250
2   80000          0.417
3   30000          0.000
4  150000          1.000


In [38]:
df_scale

,age,salary,age_standard,salary_minmax
0,25,40000,-0.788742,0.083333
1,30,60000,-0.172537,0.250000
2,35,80000,0.443667,0.416667
3,22,30000,-1.158465,0.000000
4,45,150000,1.676077,1.000000


In [39]:
# ── RobustScaler ────────────────────────────────────────────────────────────
# Uses the MEDIAN and IQR instead of mean/std.
# This makes it resistant to outliers (like the ₨150,000 salary above).
# Formula: x_scaled = (x - median) / IQR

rb = RobustScaler()
df_scale['salary_robust'] = rb.fit_transform(df_scale[['salary']])

print("Salary after RobustScaler (outlier-resistant):")
print(df_scale[['salary', 'salary_robust']].round(3))
print()
print("📌 Important rule:")
print("   ALWAYS fit the scaler on TRAINING data only.")
print("   Then use .transform() (NOT .fit_transform()) on the test data.")
print("   Otherwise you 'leak' test information into training — data leakage!")


Salary after RobustScaler (outlier-resistant):
   salary  salary_robust
0   40000          -0.50
1   60000           0.00
2   80000           0.50
3   30000          -0.75
4  150000           2.25

📌 Important rule:
   ALWAYS fit the scaler on TRAINING data only.
   Then use .transform() (NOT .fit_transform()) on the test data.
   Otherwise you 'leak' test information into training — data leakage!


---
## 🎯 Section 3 — Feature Selection

### Why do we need feature selection?

More features ≠ better model. Irrelevant features:
- Add noise that confuses the model
- Slow down training
- Can hurt accuracy (the **curse of dimensionality**)

We want to keep only the **most informative** features.

We'll use the **Breast Cancer dataset** (built into sklearn) — 569 samples, 30 features.


In [ ]:
# ── Load the breast cancer dataset ──────────────────────────────────────────
data = load_breast_cancer()
X, y = data.data, data.target

print(f"Dataset shape: {X.shape}")
print(f"  → {X.shape[0]} samples (patients)")
print(f"  → {X.shape[1]} features (measurements)")
print()
print("Feature names (first 10):", data.feature_names[:10].tolist())
print()
print("Target: 0 = malignant (cancer), 1 = benign (no cancer)")
print(f"Class distribution: {np.bincount(y)} → [{np.bincount(y)[0]} malignant, {np.bincount(y)[1]} benign]")


### 3.1 — Method 1: Statistical Tests (SelectKBest)

`SelectKBest` ranks every feature using a statistical test (`f_classif` = F-test for classification)  
and keeps the top **k** features.

The F-test measures: *"How much does this feature's value change between classes?"*  
High F-score → feature is informative.


In [ ]:
# ── SelectKBest with F-test ─────────────────────────────────────────────────
# k=10 means "keep the 10 best features"

selector_f = SelectKBest(f_classif, k=10)
X_f = selector_f.fit_transform(X, y)

print(f"Original shape: {X.shape}  →  After SelectKBest: {X_f.shape}")
print()

# Show which features were selected and their F-scores
scores = selector_f.scores_
feature_names = data.feature_names
top10_idx = scores.argsort()[-10:][::-1]   # indices of top 10 scores

print("Top 10 features by F-score:")
for rank, idx in enumerate(top10_idx, 1):
    print(f"  {rank:2}. {feature_names[idx]:<35} F-score = {scores[idx]:.1f}")


### 3.2 — Method 2: Recursive Feature Elimination (RFE)

RFE works by:
1. Train a model on all features
2. Remove the **least important** feature
3. Repeat until you have the desired number of features

It's slower but smarter — it considers feature interactions.


In [ ]:
# ── RFE ─────────────────────────────────────────────────────────────────────
# We use a small Random Forest (50 trees) as the estimator inside RFE

rfe = RFE(
    estimator=RandomForestClassifier(n_estimators=50, random_state=42),
    n_features_to_select=10
)
X_rfe = rfe.fit_transform(X, y)

print(f"Original shape: {X.shape}  →  After RFE: {X_rfe.shape}")
print()

# Which features did RFE select?
selected_mask = rfe.support_   # True/False for each feature
selected_names = [name for name, keep in zip(data.feature_names, selected_mask) if keep]
print("Features selected by RFE:")
for i, name in enumerate(selected_names, 1):
    print(f"  {i:2}. {name}")


### 3.3 — Method 3: Feature Importance from a Tree Model

Random Forests automatically calculate how much each feature reduces "impurity" (disorder) in the data.  
We can use this importance score to drop features below a threshold.


In [ ]:
# ── Tree-based Feature Importance ───────────────────────────────────────────
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X, y)

# SelectFromModel drops features whose importance < threshold
# threshold='median' → keep the top 50% most important features
selector_tree = SelectFromModel(rf, threshold='median')
X_tree = selector_tree.transform(X)

print(f"Original shape: {X.shape}  →  After SelectFromModel: {X_tree.shape}")
print()

# Show all feature importances sorted
importances = rf.feature_importances_
sorted_idx  = importances.argsort()[::-1]

print("All features ranked by importance (higher = more useful):")
for rank, idx in enumerate(sorted_idx, 1):
    bar = "█" * int(importances[idx] * 200)
    print(f"  {rank:2}. {data.feature_names[idx]:<35} {importances[idx]:.4f}  {bar}")


---
## ⚖️ Section 4 — SMOTE: Handling Imbalanced Data

### The problem: Class Imbalance

Imagine a fraud detection dataset with:
- 99,000 legitimate transactions
- 1,000 fraudulent transactions

A model that **always predicts "Legitimate"** would be **99% accurate** — but completely useless!

The model is biased because it barely sees fraud examples during training.

### The solution: SMOTE

**SMOTE** (Synthetic Minority Over-sampling Technique) creates **artificial new examples**  
for the minority class by interpolating between existing minority examples.

> 💡 SMOTE doesn't just copy examples — it creates *new* points between real ones.

```
Real fraud example A: [amount=500, hour=2]
Real fraud example B: [amount=300, hour=3]
SMOTE creates:        [amount=400, hour=2.5]  ← a new synthetic fraud example
```


In [ ]:
# ── Create a synthetic imbalanced dataset ──────────────────────────────────
# We simulate a fraud detection problem.
# Class 0 = Legitimate (1000 samples), Class 1 = Fraud (50 samples)

np.random.seed(42)

# Legitimate transactions: normally distributed around mean=[50, 5]
X_legit = np.random.randn(1000, 4) * [10, 2, 5, 1] + [50, 5, 100, 3]
y_legit = np.zeros(1000, dtype=int)   # 0 = legitimate

# Fraudulent transactions: different distribution
X_fraud = np.random.randn(50, 4)  * [5,  3, 2, 2] + [80, 1, 20,  8]
y_fraud = np.ones(50, dtype=int)   # 1 = fraud

# Combine into one dataset
X_imbalanced = np.vstack([X_legit, X_fraud])
y_imbalanced = np.hstack([y_legit, y_fraud])

print("Imbalanced dataset created!")
print(f"  Total samples : {len(y_imbalanced)}")
print(f"  Legitimate (0): {(y_imbalanced == 0).sum()} samples")
print(f"  Fraud      (1): {(y_imbalanced == 1).sum()} samples")
print(f"  Imbalance ratio: {(y_imbalanced==0).sum() / (y_imbalanced==1).sum():.0f}:1")


In [ ]:
# ── Train/Test Split & Scaling ──────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_imbalanced, y_imbalanced,
    test_size=0.2,     # 20% for testing
    random_state=42,
    stratify=y_imbalanced   # keep the class ratio the same in train and test
)

# Scale features (fit ONLY on training data!)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit + transform on train
X_test_scaled  = scaler.transform(X_test)         # transform only on test

print(f"Training set: {X_train_scaled.shape}  |  Fraud in train: {y_train.sum()}")
print(f"Test set    : {X_test_scaled.shape}  |  Fraud in test : {y_test.sum()}")


In [ ]:
# ── Without SMOTE ──────────────────────────────────────────────────────────
print("=" * 55)
print("MODEL 1: Random Forest WITHOUT SMOTE")
print("=" * 55)

rf_no_smote = RandomForestClassifier(n_estimators=100, random_state=42)
rf_no_smote.fit(X_train_scaled, y_train)
y_pred_no_smote = rf_no_smote.predict(X_test_scaled)

# classification_report shows Precision, Recall, F1 for each class
print(classification_report(y_test, y_pred_no_smote, target_names=['Legitimate', 'Fraud']))
print()
print("📌 Watch the 'Fraud' row — recall is often very low without SMOTE")
print("   Recall = out of all real frauds, how many did we catch?")


In [ ]:
# ── Apply SMOTE ─────────────────────────────────────────────────────────────
# SMOTE generates synthetic minority class samples ONLY in the training set
# NEVER apply SMOTE to the test set — that would be data leakage!

smote = SMOTE(
    random_state=42,
    k_neighbors=5    # each new sample is created using 5 nearest neighbours
)

X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print("After applying SMOTE to training data:")
print(f"  Before SMOTE → Legitimate: {(y_train==0).sum()}, Fraud: {(y_train==1).sum()}")
print(f"  After  SMOTE → Legitimate: {(y_train_smote==0).sum()}, Fraud: {(y_train_smote==1).sum()}")
print()
print("SMOTE created synthetic fraud examples until both classes are balanced! ✅")


In [ ]:
# ── With SMOTE ─────────────────────────────────────────────────────────────
print("=" * 55)
print("MODEL 2: Random Forest WITH SMOTE")
print("=" * 55)

rf_smote = RandomForestClassifier(n_estimators=100, random_state=42)
rf_smote.fit(X_train_smote, y_train_smote)   # train on the balanced data
y_pred_smote = rf_smote.predict(X_test_scaled)  # test on ORIGINAL (unbalanced) test set

print(classification_report(y_test, y_pred_smote, target_names=['Legitimate', 'Fraud']))


In [ ]:
# ── Side-by-side Comparison ─────────────────────────────────────────────────
from sklearn.metrics import f1_score, recall_score, precision_score

def get_metrics(y_true, y_pred):
    return {
        'Precision (Fraud)': precision_score(y_true, y_pred, pos_label=1),
        'Recall    (Fraud)': recall_score(   y_true, y_pred, pos_label=1),
        'F1        (Fraud)': f1_score(       y_true, y_pred, pos_label=1),
    }

metrics_no = get_metrics(y_test, y_pred_no_smote)
metrics_sm = get_metrics(y_test, y_pred_smote)

print("╔══════════════════════════╦════════════════╦════════════════╗")
print("║ Metric                   ║  Without SMOTE ║   With SMOTE   ║")
print("╠══════════════════════════╬════════════════╬════════════════╣")
for key in metrics_no:
    diff = metrics_sm[key] - metrics_no[key]
    sign = "+" if diff >= 0 else ""
    print(f"║ {key}  ║ {metrics_no[key]:>12.3f}   ║ {metrics_sm[key]:>12.3f}   ║  ({sign}{diff:.3f})")
print("╚══════════════════════════╩════════════════╩════════════════╝")
print()
print("📌 Recall improves most — the model now catches more real fraud cases.")
print("📌 Precision may drop slightly — it makes a few more false alarms.")
print("   In fraud detection, catching more fraud (recall) is usually worth it!")


---
## ✅ Day 1 Summary — What Did We Learn?

| Topic | Key Takeaway |
|-------|-------------|
| **Label Encoding** | Converts ordered text to integers (Bachelors=0, Masters=1, PhD=2) |
| **One-Hot Encoding** | Creates binary columns for each category — no implied order |
| **Ordinal Encoding** | You manually define the order |
| **StandardScaler** | Mean=0, Std=1 — best for most ML algorithms |
| **MinMaxScaler** | Squashes to [0,1] — best for neural networks |
| **RobustScaler** | Uses median — best when data has outliers |
| **SelectKBest** | Statistical test to pick top-k features |
| **RFE** | Iteratively removes weakest features |
| **Tree Importance** | Uses Random Forest importances to drop low-value features |
| **SMOTE** | Creates synthetic minority class examples to fix imbalanced data |

---

### 🟢 Practice Questions for Today

1. Why can't you use One-Hot Encoding for a column with 1000 unique categories? What would you use instead?
2. You have ages: `[10, 12, 11, 400, 9]`. The 400 is an error. Which scaler is most robust to this outlier? Why?
3. Add a `log_transaction_amount` feature (use `np.log1p`) to the fraud dataset and check if it improves the model.

---
*Week 3, Day 1 | Applied AI & LLM Engineer | Edversity*
